In [0]:
from pyspark.sql.functions import from_json, col, to_timestamp, year, month, dayofmonth, round
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
# Crear esquema silver si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.hd_lima_air_pollution (
    dt_ts TIMESTAMP COMMENT 'Date and time, Unix UTC',
    id_ubigeo STRING COMMENT 'Código único de distrito (UBIGEO)',
    nom_distrito STRING COMMENT 'Nombre del distrito',
    nom_provincia STRING COMMENT 'Nombre de la provincia',
    nom_departamento STRING COMMENT 'Nombre del departamento',
    latitud DOUBLE COMMENT 'Latitud del distrito',
    longitud DOUBLE COMMENT 'Longitud del distrito',
    index_air_quality INT COMMENT 'Air Quality Index (1=Good, 5=Very Poor)',
    conc_co DOUBLE COMMENT 'Concentration of CO (Carbon monoxide), μg/m3',
    conc_no DOUBLE COMMENT 'Concentration of NO (Nitrogen monoxide), μg/m3',
    conc_no2 DOUBLE COMMENT 'Concentration of NO2 (Nitrogen dioxide), μg/m3',
    conc_o3 DOUBLE COMMENT 'Concentration of O3 (Ozone), μg/m3',
    conc_so2 DOUBLE COMMENT 'Concentration of SO2 (Sulphur dioxide), μg/m3',
    conc_pm2_5 DOUBLE COMMENT 'Concentration of PM2.5 (Fine particles matter), μg/m3',
    conc_pm10 DOUBLE COMMENT 'Concentration of PM10 (Coarse particulate matter), μg/m3',
    conc_nh3 DOUBLE COMMENT 'Concentration of NH3 (Ammonia), μg/m3',
    year INT COMMENT 'Año derivado de dt',
    month INT COMMENT 'Mes derivado de dt',
    day INT COMMENT 'Día derivado de dt'
) USING DELTA;

In [0]:
# Definir esquemas para parsear los JSON internos
main_schema = StructType([
    StructField("aqi", StringType(), True)
])

components_schema = StructType([
    StructField("co", StringType(), True),
    StructField("no", StringType(), True),
    StructField("no2", StringType(), True),
    StructField("o3", StringType(), True),
    StructField("so2", StringType(), True),
    StructField("pm2_5", StringType(), True),
    StructField("pm10", StringType(), True),
    StructField("nh3", StringType(), True)
])

# Leer Bronze
df_bronze = spark.table("workspace.bronze.hist_air_pollution")

# Parsear JSON y aplicar transformaciones
df_silver = (
    df_bronze
    .withColumn("main_parsed", from_json(col("main"), main_schema))
    .withColumn("components_parsed", from_json(col("components"), components_schema))
    # Renombrar y castear
    .withColumn("index_air_quality", col("main_parsed.aqi").cast("int"))
    .withColumn("conc_co", round(col("components_parsed.co"),2).cast("double"))
    .withColumn("conc_no", round(col("components_parsed.no"),2).cast("double"))
    .withColumn("conc_no2", round(col("components_parsed.no2"),2).cast("double"))
    .withColumn("conc_o3", round(col("components_parsed.o3"),2).cast("double"))
    .withColumn("conc_so2", round(col("components_parsed.so2"),2).cast("double"))
    .withColumn("conc_pm2_5", round(col("components_parsed.pm2_5"),2).cast("double"))
    .withColumn("conc_pm10", round(col("components_parsed.pm10"),2).cast("double"))
    .withColumn("conc_nh3", round(col("components_parsed.nh3"),2).cast("double"))
    # Derivar fecha desde dt
    .withColumn("dt_ts", to_timestamp(col("dt").cast("long")))
    .withColumn("year", year(col("dt_ts")))
    .withColumn("month", month(col("dt_ts")))
    .withColumn("day", dayofmonth(col("dt_ts")))
    # Renombrar campos de ubicación
    .withColumnRenamed("ubigeo", "id_ubigeo")
    .withColumnRenamed("distrito", "nom_distrito")
    .withColumnRenamed("provincia", "nom_provincia")
    .withColumnRenamed("departamento", "nom_departamento")
    # Castear lat/long
    .withColumn("latitud", col("latitud").cast("double"))
    .withColumn("longitud", col("longitud").cast("double"))
    .drop("main_parsed", "components_parsed","main","components","dt")
)


In [0]:
# Guardar en tabla Silver
df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.hd_lima_air_pollution")